# Verifying "Where Credit Enters a Computation"

Two self-contained checks of the paper's claims, kept short.

**Part 1 - toy model** (Section 5): the exact transition-matrix executor, where every theorem is an algebraic identity we can check to machine precision or by direct simulation - no training needed to "believe" the mechanism.

**Part 2 - full model** (Section 4/6): a small causal Transformer trained on the paper's matched-supervision conditions (`outcome`, `answer_first`, `process`, `corrupted`), reproducing the central empirical finding that only valid pre-answer process supervision teaches execution.

Everything below runs on CPU/CUDA in a couple of minutes total. Constants are set for a fast demonstration; each part notes what to change to approach the paper's exact numbers.

In [3]:
import torch, torch.nn as nn, numpy as np, time
torch.manual_seed(0); rng = np.random.default_rng(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cuda


## Part 1 - Toy transition-matrix executor

A random depth-`D`, `K`-state circuit; `P_t` is the model's learned row-stochastic kernel for the gate used at step `t`. Everything here is exact algebra (Section 5), so we check identities directly rather than by training.

### Theorem 1 - the exact forward-backward factorization

The raw (unconstrained) derivative is the bilinear identity `-d(ell_out)/dP_t = q_{t-1} beta_t^T / [q_D]_y` (eq. 22). Theorem 1 itself is the stronger claim about that derivative *restricted to the tangent space* `T` of row-stochastic matrices (only directions with zero row-sum are feasible perturbations): `Pi_T[-d(ell_out)/dP_t] = q_{t-1} (Pi*beta_t)^T / [q_D]_y` (eq. 10), i.e. `beta_t` gets centered too. We build genuinely row-stochastic `P_t` (softmax rows, checked to have strictly positive entries), take the raw autograd gradient, project *it* onto `T` (subtract each row's mean), and compare against the eq. (10) formula with `beta_t` itself centered -- not the unprojected eq. (22) form.

In [4]:
K, D = 8, 4
logits = [torch.randn(K, K) for _ in range(D)]
Ps = [torch.softmax(l, dim=-1).detach().requires_grad_(True) for l in logits]   # genuinely row-stochastic leaves
s0, y = 2, 5

qs = [torch.zeros(K)]; qs[0][s0] = 1.0
for P in Ps:
    qs.append(qs[-1] @ P)                                          # qs[k] = q_k  (forward states)
loss = -torch.log(qs[-1][y])
autograds = torch.autograd.grad(loss, Ps)                          # raw ambient gradient (eq. 22)

e_y = torch.zeros(K); e_y[y] = 1.0
betas = {D: e_y}
for k in range(D - 1, 0, -1):
    betas[k] = (Ps[k] @ betas[k + 1]).detach()                     # beta_t = P_{t+1}...P_D e_y
center = lambda v: v - v.mean()                                     # Pi: projection onto 1^perp

qD_y = qs[-1][y].detach()
print("min entry across all P_t:", min(P.min().item() for P in Ps), "(strictly positive => genuinely row-stochastic)")

max_err = 0.0
for j in range(D):
    G_T = (-autograds[j]) - (-autograds[j]).mean(dim=1, keepdim=True)         # Pi_T[-d(ell_out)/dP_t]: eq. 10 LHS
    manual = torch.outer(qs[j].detach(), center(betas[j + 1])) / qD_y          # q_{t-1}(Pi*beta_t)^T/[q_D]_y: eq. 10 RHS
    max_err = max(max_err, (G_T - manual).abs().max().item())
print(f"Theorem 1: max |Pi_T[-d(ell_out)/dP_t] - q_(t-1)(Pi*beta_t)^T/[q_D]_y| = {max_err:.2e}  (exact identity, eq. 10)")
assert max_err < 1e-5

min entry across all P_t: 0.004821724258363247 (strictly positive => genuinely row-stochastic)
Theorem 1: max |Pi_T[-d(ell_out)/dP_t] - q_(t-1)(Pi*beta_t)^T/[q_D]_y| = 2.87e-07  (exact identity, eq. 10)


### Theorem 2 - zero information about the executor at complete mixing

Theorem 2 is a claim about the *entire law* of the outcome gradient, not just its alignment with the executor direction: at `P_g = U`, the gradient is a deterministic function of the pair (final gate, terminal state), and by Lemma 4 that pair's joint law is `Unif(gates) x Unif(states)` for *any* reversible executor -- the specific permutations never enter. With `K=8, M=4, D=4` the entire population is only `K*M^D = 2048` examples, small enough to enumerate exactly rather than estimate from samples. So we enumerate every `(s0, gate sequence)` pair for two totally unrelated executors and require the resulting `(final gate, terminal state)` histograms to be bit-for-bit identical -- an exact finite-population verification, not statistical evidence.

In [5]:
import itertools

def make_executor(K, M, seed):
    g = torch.Generator().manual_seed(seed)
    return [torch.randperm(K, generator=g) for _ in range(M)]

def exact_gate_state_hist(perms, K, M, D):
    counts = torch.zeros(M, K, dtype=torch.long)
    for s0 in range(K):                                        # every (s0, gate sequence) pair, exactly once:
        for gate_seq in itertools.product(range(M), repeat=D):  # this IS the population, not a sample of it
            s = s0
            for gidx in gate_seq: s = perms[gidx][s].item()
            counts[gate_seq[-1], s] += 1
    return counts

execA, execB = make_executor(K, 4, 1), make_executor(K, 4, 2)     # two totally different, unrelated executors
histA = exact_gate_state_hist(execA, K, 4, D)
histB = exact_gate_state_hist(execB, K, 4, D)

print(f"Theorem 2: enumerated all K*M^D = {K * 4**D} examples per executor")
print(f"  histA unique cell counts: {histA.unique().tolist()}  (exactly uniform: {K*4**D}/{4*K} = {K*4**D//(4*K)} per cell)")
assert torch.equal(histA, histB)
assert (histA == histA[0, 0]).all()
print("EXACT: the two executors' (final gate, terminal state) histograms are bit-for-bit identical and exactly uniform --")
print("the gradient's law does not depend on which executor generated it, not merely orthogonal to it.")

Theorem 2: enumerated all K*M^D = 2048 examples per executor
  histA unique cell counts: [64]  (exactly uniform: 2048/32 = 64 per cell)
EXACT: the two executors' (final gate, terminal state) histograms are bit-for-bit identical and exactly uniform --
the gradient's law does not depend on which executor generated it, not merely orthogonal to it.


### Theorem 3 - suppression on a whole neighborhood, not just at one point

`P_g(alpha) = (1-alpha)*U + alpha*R_g` for a permutation `R_g` is a genuine row-stochastic matrix for every `alpha in [0,1]` (a convex combination of two nonnegative row-stochastic matrices is nonnegative and row-stochastic) -- unlike scaling an unconstrained random perturbation, which can and does push entries negative. We fix the *direction* `R_g` once and only vary `alpha` (so the log-log fit isn't confounded by re-randomizing the direction at every point), and check both that every `P_g(alpha)` is a valid transition matrix and that the fitted slope of `conditional credit` vs `phi = alpha * max_g||R_g-U||_op` matches the predicted exponent `D-1`.

In [6]:
def project_C(M):
    M = M - M.mean(dim=1, keepdim=True)      # onto T: rows sum to 0
    return M - M.mean(dim=0, keepdim=True)    # drop the R part: columns re-centered too

def conditional_credit(Ps, s0, y):
    D = len(Ps)
    qs = [torch.zeros(K)]; qs[0][s0] = 1.0
    for P in Ps: qs.append(qs[-1] @ P)
    e_y = torch.zeros(K); e_y[y] = 1.0
    betas = {D: e_y}
    for k in range(D - 1, 0, -1): betas[k] = Ps[k] @ betas[k + 1]
    qD_y = qs[-1][y]
    return [project_C(torch.outer(qs[j], betas[j + 1]) / qD_y).norm().item() for j in range(D)]

U = torch.full((K, K), 1.0 / K)
torch.manual_seed(3)
Rs = [torch.eye(K)[torch.randperm(K)] for _ in range(D)]      # fixed permutation directions, reused at every alpha
F_dirs = [R - U for R in Rs]                                   # each lies in C automatically (Lemma 3)
op_norm = max(torch.linalg.matrix_norm(F, ord=2).item() for F in F_dirs)

alphas = [0.4, 0.3, 0.2, 0.1, 0.05]
phis, credits = [], []
print(f"{'alpha':>6} {'phi':>7}  " + "  ".join(f"t={t+1}" for t in range(D)) + "   min P entry")
for alpha in alphas:
    Ps = [U + alpha * F for F in F_dirs]                       # (1-alpha)U + alpha*R_g: convex -> entrywise >= 0
    min_entry = min(P.min().item() for P in Ps)
    assert min_entry >= -1e-8, "not row-stochastic!"
    credit = conditional_credit(Ps, 1, 4)
    phi = alpha * op_norm
    phis.append(phi); credits.append(max(credit))
    print(f"{alpha:6.2f} {phi:7.3f}  " + "  ".join(f"{c:.2e}" for c in credit) + f"   {min_entry:.4f}")

slope = np.polyfit(np.log(phis), np.log(credits), 1)[0]
print(f"\nfitted log(credit)/log(phi) slope = {slope:.2f}   predicted D-1 = {D - 1}")
assert abs(slope - (D - 1)) < 0.5

 alpha     phi  t=1  t=2  t=3  t=4   min P entry
  0.40   0.400  4.60e-01  4.60e-01  4.60e-01  4.60e-01   0.0750
  0.30   0.300  1.91e-01  1.91e-01  1.91e-01  1.91e-01   0.0875
  0.20   0.200  5.61e-02  5.61e-02  5.61e-02  5.61e-02   0.1000
  0.10   0.100  7.00e-03  7.00e-03  7.00e-03  7.00e-03   0.1125
  0.05   0.050  8.75e-04  8.75e-04  8.75e-04  8.75e-04   0.1187

fitted log(credit)/log(phi) slope = 3.01   predicted D-1 = 3


### Theorem 4 - outcome flow is trapped for a depth-graded time

If the flow is kept purely conditional (every gradient step re-projected onto `C`, i.e. `gamma` held at 0), Theorem 3's neighborhood bound integrates into an escape-time law: `phi(t) <= 2*eps` for `t <= c * eps^{-(D-2)}`. To test this faithfully as *population* gradient flow (not a stochastic approximation to it) we enumerate the entire finite population `{(s0, gate sequence)}` once and reuse it, unchanged, at every step -- this makes the loss and its gradient exact, not a minibatch estimate. We also initialize at a genuinely row-stochastic point via the same convex trick as Theorem 3, `P_g(0) = (1-eps/c)*U + (eps/c)*R_g`, and track the minimum transition-matrix entry seen along the whole trajectory up to escape (not just at initialization) to confirm the flow never actually leaves the space of valid transition matrices before "escape" is declared.

In [7]:
Kx, Mx = 6, 4
U_ = torch.full((Kx, Kx), 1.0 / Kx)

def project_C_batched(F):
    F = F - F.mean(dim=-1, keepdim=True)      # onto T: rows sum to 0 (works for a single matrix or a stack)
    return F - F.mean(dim=-2, keepdim=True)   # drop the R part

def full_population(Dv):
    s0 = torch.arange(Kx).repeat_interleave(Mx ** Dv)
    gate_idx = torch.tensor(list(itertools.product(range(Mx), repeat=Dv))).repeat(Kx, 1)
    return s0, gate_idx                        # every (s0, gate sequence) pair, exactly once -- the whole population

def escape_time(Dv, eps, perms, seed, lr=0.5, max_steps=60000):
    g = torch.Generator().manual_seed(seed)
    R = torch.eye(Kx)[torch.stack([torch.randperm(Kx, generator=g) for _ in range(Mx)])]  # random permutation/gate
    c = max(torch.linalg.matrix_norm(R[i] - U_, ord=2).item() for i in range(Mx))
    Fp = (eps * (R - U_) / c).clone().requires_grad_(True)   # (1-eps/c)U + (eps/c)R_g: exactly row-stochastic at t=0
    s0, gate_idx = full_population(Dv)
    min_entry_seen = 1.0
    for step in range(1, max_steps + 1):
        s = s0.clone()
        for t in range(Dv): s = perms[gate_idx[:, t], s]
        P = U_ + Fp
        min_entry_seen = min(min_entry_seen, P.min().item())
        q = torch.eye(Kx)[s0]
        for t in range(Dv): q = torch.einsum('nk,nkl->nl', q, P[gate_idx[:, t]])
        loss = -torch.log(q.gather(1, s[:, None]).squeeze(1) + 1e-12).mean()   # exact population loss, no sampling
        gr, = torch.autograd.grad(loss, Fp)
        with torch.no_grad():
            Fp -= lr * project_C_batched(gr)               # keep the flow purely conditional: maintains gamma == 0
            phi = max(torch.linalg.matrix_norm(f, ord=2).item() for f in Fp)
        if phi > 2 * eps: return step, min_entry_seen
    return max_steps, min_entry_seen

perms4 = torch.stack([torch.randperm(Kx) for _ in range(Mx)])
eps_grid = [0.15, 0.10, 0.07, 0.05]
esc = {}
for Dv in [3, 4]:
    for eps in eps_grid:
        runs = [escape_time(Dv, eps, perms4, seed) for seed in [1, 2]]
        steps, min_entries = [r[0] for r in runs], [r[1] for r in runs]
        esc[(Dv, eps)] = sum(steps) / len(steps)
        print(f"D={Dv}  eps={eps:.2f}  escape steps (2 seeds) = {steps}  min P entry seen = {[f'{m:.3f}' for m in min_entries]}")
        assert min(min_entries) > 0, "the flow left the space of valid transition matrices before escape!"

for Dv in [3, 4]:
    xs = np.log(1 / np.array(eps_grid))
    ys = np.log([esc[(Dv, e)] for e in eps_grid])
    slope = np.polyfit(xs, ys, 1)[0]
    print(f"D={Dv}: measured log(escape)/log(1/eps) slope = {slope:.2f}   predicted (gamma=0 bound) = D-2 = {Dv-2}")
    assert abs(slope - (Dv - 2)) < 1.0

D=3  eps=0.15  escape steps (2 seeds) = [49, 35]  min P entry seen = ['0.056', '0.084']
D=3  eps=0.10  escape steps (2 seeds) = [71, 51]  min P entry seen = ['0.094', '0.112']
D=3  eps=0.07  escape steps (2 seeds) = [101, 71]  min P entry seen = ['0.116', '0.129']
D=3  eps=0.05  escape steps (2 seeds) = [141, 99]  min P entry seen = ['0.130', '0.140']
D=4  eps=0.15  escape steps (2 seeds) = [242, 314]  min P entry seen = ['0.077', '0.075']
D=4  eps=0.10  escape steps (2 seeds) = [539, 702]  min P entry seen = ['0.107', '0.106']
D=4  eps=0.07  escape steps (2 seeds) = [1097, 1430]  min P entry seen = ['0.125', '0.124']
D=4  eps=0.05  escape steps (2 seeds) = [2147, 2800]  min P entry seen = ['0.137', '0.136']
D=3: measured log(escape)/log(1/eps) slope = 0.96   predicted (gamma=0 bound) = D-2 = 1
D=4: measured log(escape)/log(1/eps) slope = 1.99   predicted (gamma=0 bound) = D-2 = 2


### Theorem 5 - process supervision converges in closed-form finite time

Under process supervision, gradient flow on each row's correct-destination probability obeys `p_dot = f_g/(K-1) * p*(1-p)^2` (eq. 16), which integrates in closed form (eq. 17). We check that the closed-form hitting time for `p=0.90` matches direct numerical integration of the ODE.

In [8]:
K_reg, f_g = 8, 1.0
F = lambda p: torch.log(torch.tensor(p / (1 - p))) + 1 / (1 - p)
t_hit = (K_reg - 1) / f_g * (F(0.9) - F(1.0 / K_reg)).item()     # closed form, eq. (16)-(17)

p, dt, t = 1.0 / K_reg, 1e-3, 0.0
while t < t_hit:
    p += dt * (f_g / (K_reg - 1)) * p * (1 - p) ** 2
    t += dt
print(f"Theorem 5: closed-form hitting time for p=0.90 is t={t_hit:.2f}; "
      f"numerically integrating the flow to that time gives p={p:.4f}.")
assert abs(p - 0.9) < 0.01

Theorem 5: closed-form hitting time for p=0.90 is t=91.00; numerically integrating the flow to that time gives p=0.9000.


### Same-parameter training dynamics (cf. Table 3)

Two copies of the same randomly initialized `{P_g}` are trained from the same data - one on outcome loss, one on process loss. Process supervision recovers the true gate table almost immediately; outcome supervision does not move within the same budget (Theorem 4's trapping).

In [9]:
M_gates, B = 6, 256
perms = torch.stack([torch.randperm(K) for _ in range(M_gates)])       # true executor {T_g}

def sample_batch(n):
    s0 = torch.randint(0, K, (n,)); gates = torch.randint(0, M_gates, (n, D))
    states = [s0]
    for t in range(D): states.append(perms[gates[:, t], states[-1]])
    return gates, torch.stack(states, dim=1)

def losses(logits, gates, states):
    P = torch.softmax(logits, dim=-1)                                 # (M,K,K)
    q = torch.eye(K)[states[:, 0]]                                    # (n,K)
    for t in range(D):
        Pt = P[gates[:, t]]                                           # (n,K,K)
        q = torch.einsum('nk,nkl->nl', q, Pt)
    out_loss = -torch.log(q.gather(1, states[:, -1:]).squeeze(1) + 1e-12).mean()
    Pt_all = P[gates]                                                  # (n,D,K,K)
    idx_i, idx_j = states[:, :-1], states[:, 1:]
    picked = Pt_all.gather(2, idx_i.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, 1, K)).squeeze(2)
    proc_p = picked.gather(2, idx_j.unsqueeze(-1)).squeeze(-1)
    proc_loss = -torch.log(proc_p + 1e-12).mean()
    return out_loss, proc_loss, q

def gate_recovery(logits): return (torch.softmax(logits, -1).argmax(-1) == perms).float().mean().item() * 100
def terminal_acc(logits):
    g, s = sample_batch(2000); _, _, q = losses(logits, g, s)
    return (q.argmax(1) == s[:, -1]).float().mean().item() * 100

init = torch.randn(M_gates, K, K) * 0.05
out_logits, proc_logits = init.clone().requires_grad_(True), init.clone().requires_grad_(True)
opt_out, opt_proc = torch.optim.Adam([out_logits], lr=0.1), torch.optim.Adam([proc_logits], lr=0.1)

print(f"{'step':>5} {'out gate%':>10} {'out term%':>10} {'proc gate%':>11} {'proc term%':>11}")
for step in range(41):
    if step % 10 == 0:
        print(f"{step:5d} {gate_recovery(out_logits):10.2f} {terminal_acc(out_logits):10.2f} "
              f"{gate_recovery(proc_logits):11.2f} {terminal_acc(proc_logits):11.2f}")
    gates, states = sample_batch(B)
    ol, _, _ = losses(out_logits, gates, states); opt_out.zero_grad(); ol.backward(); opt_out.step()
    _, pl, _ = losses(proc_logits, gates, states); opt_proc.zero_grad(); pl.backward(); opt_proc.step()

 step  out gate%  out term%  proc gate%  proc term%
    0      12.50      12.60       12.50       13.15
   10      10.42      13.40      100.00      100.00
   20      12.50      12.45      100.00      100.00
   30      12.50      12.45      100.00      100.00
   40       6.25      12.00      100.00      100.00


### Theorem 6 - landscape dichotomy in a one-layer causal Transformer

At the marginal-collapse point `theta_pi` (Definition 1: `W_E=W_U=W_O=W_2=0`, `b=log(pi)`), Lemma 6 says the network ignores its input, predicts the constant `pi`, and sits at a stationary point of both objectives. Theorem 6 says the Hessian's `(W_U, W_E)` cross-block is `-Cov(y_n, tau_n) tensor I`: `theta_pi` is a local minimum when the supervised target is uncorrelated with the current token, and a strict saddle when it isn't. With a small enough vocabulary and sequence length (`V=4, L=4`) the full population of sequences (`V^L=256`, or `V^(2L)=65536` token/target pairs for the independent case) is cheap to enumerate exactly, so we compute the exact-population Hessian rather than a Monte Carlo estimate -- this is what lets `Cov(y,tau)=0` land at exactly (float-precision) zero curvature instead of a small negative finite-sample residual.

In [10]:
V, d, L = 4, 4, 4     # small enough that the FULL population of length-L sequences is enumerable exactly

pos = torch.randn(L, d) * 0.1                                    # positional embedding: "arbitrary" per Def. 1
WQ, WK, WV, W1 = (torch.randn(d, d) * 0.3 for _ in range(4))     # also "arbitrary" -- W_O, W_2 are pinned to 0
causal_mask = torch.triu(torch.ones(L, L), 1).bool()

def forward(tokens, WE, WO, W2, WU, b):
    tau = nn.functional.one_hot(tokens, V).float()
    x = tau @ WE + pos
    Q, Kk, Vv = x @ WQ, x @ WK, x @ WV
    scores = (Q @ Kk.transpose(-1, -2) / d ** 0.5).masked_fill(causal_mask, -1e9)
    y = x + (torch.softmax(scores, -1) @ Vv) @ WO
    z = y + nn.functional.relu(y @ W1) @ W2
    return z @ WU + b

all_seqs = torch.tensor(list(itertools.product(range(V), repeat=L)))   # every length-L sequence, V^L = 256 of them

def exact_population(target_mode):
    if target_mode == 'copy':
        return all_seqs, all_seqs.clone()                              # target = current token: Cov(y,tau) != 0
    n = len(all_seqs)                                                  # 'independent': every (token-seq, target-seq)
    return all_seqs.repeat_interleave(n, dim=0), all_seqs.repeat(n, 1) # pair, exactly once -> Cov(y,tau) == 0 exactly

pi = torch.full((V,), 1.0 / V)                                          # exact marginal (uniform by construction)
H_pi = -(pi * pi.log()).sum().item()

# ---- Lemma 6 at theta_pi, exact population (zero sampling noise anywhere) ----
tokens, targets = exact_population('copy')
WE0, WO0, W20, WU0 = torch.zeros(V, d), torch.zeros(d, d), torch.zeros(d, d), torch.zeros(d, V)
b0 = torch.log(pi)
for p in (WE0, WO0, W20, WU0, b0): p.requires_grad_(True)
logits = forward(tokens, WE0, WO0, W20, WU0, b0)
pred = torch.softmax(logits, -1)
loss = nn.functional.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
print(f"Lemma 6 (exact population): prediction == pi everywhere: {torch.allclose(pred, pi.expand_as(pred), atol=1e-6)}")
print(f"Lemma 6 (exact population): loss = {loss.item():.8f} vs H(pi) = {H_pi:.8f}")
grads = torch.autograd.grad(loss, [WE0, WO0, W20, WU0, b0])
print(f"Lemma 6 (exact population): max |grad| at theta_pi = {max(g.abs().max().item() for g in grads):.2e}  (exact zero, up to float precision)")

# ---- Theorem 6: exact-population Hessian of the (W_U, W_E) block ----
def hessian_min_eig(target_mode):
    tokens, targets = exact_population(target_mode)
    b, WO, W2 = torch.log(pi), torch.zeros(d, d), torch.zeros(d, d)
    def loss_of(v):
        WE, WU = v[:V * d].view(V, d), v[V * d:].view(d, V)
        logits = forward(tokens, WE, WO, W2, WU, b)
        return nn.functional.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))
    Hm = torch.autograd.functional.hessian(loss_of, torch.zeros(V * d + d * V))
    eigvals, eigvecs = torch.linalg.eigh(Hm)
    v_min = eigvecs[:, 0]
    we_mass = v_min[:V * d].pow(2).sum().item() / v_min.pow(2).sum().item()
    return eigvals[0].item(), we_mass

lam_indep, mass_indep = hessian_min_eig('independent')     # Cov(y,tau)=0  -> Theorem 6(i): local minimum
lam_copy, mass_copy = hessian_min_eig('copy')              # Cov(y,tau)!=0 -> Theorem 6(ii): strict saddle
print(f"\nTheorem 6 (exact population): min eigenvalue of the (W_U,W_E) Hessian block")
print(f"  Cov(y,tau)=0  (independent targets): lambda_min = {lam_indep:.3e}  -> ~0 (local minimum)")
print(f"  Cov(y,tau)!=0 (copy-current-token)  : lambda_min = {lam_copy:.3e}  -> strict saddle")
print(f"  fraction of the escape eigenvector's mass on W_E: independent {mass_indep:.2f}, copy {mass_copy:.2f} (paper: ~50/50 split with W_U)")
assert abs(lam_indep) < 1e-5
assert lam_copy < -1e-3

Lemma 6 (exact population): prediction == pi everywhere: True
Lemma 6 (exact population): loss = 1.38629472 vs H(pi) = 1.38629436
Lemma 6 (exact population): max |grad| at theta_pi = 2.07e-08  (exact zero, up to float precision)



Theorem 6 (exact population): min eigenvalue of the (W_U,W_E) Hessian block
  Cov(y,tau)=0  (independent targets): lambda_min = -1.884e-08  -> ~0 (local minimum)
  Cov(y,tau)!=0 (copy-current-token)  : lambda_min = -2.499e-01  -> strict saddle
  fraction of the escape eigenvector's mass on W_E: independent 0.00, copy 0.50 (paper: ~50/50 split with W_U)


### Proposition 1 - the reliability boundary

Under m-fold corrupted-successor competition, the reduced local model's logit margin `z` obeys gradient flow on `L(z) = -a*log(sigma(z)) - b*log(1-sigma(z))` with `a=rho, b=(1-rho)*nu_max` (eq. 7), and the claim is that the flow's equilibrium changes sign exactly at `rho_c = 1/(m+1)`. We don't just evaluate the closed-form formula against itself (that would only check that `(1/m)/(1+1/m)` and `1/(m+1)` are the same fraction) -- we numerically integrate the actual gradient flow from `z=0` on both sides of `rho_c` for each `m` and check that the *simulated* equilibrium's sign matches the prediction and its value matches the closed form.

In [11]:
def simulate_flow(rho, nu_max, eta=1.0, steps=200000, dt=0.01):
    a, b = rho, (1 - rho) * nu_max            # eq. (7): L_rho,c(z) = -a*log(sigma(z)) - b*log(1-sigma(z))
    z = 0.0
    for _ in range(steps):
        sig = 1 / (1 + np.exp(-z))
        z += dt * (-eta * ((a + b) * sig - a))  # gradient flow on the logit margin z
    return z

for m in [1, 3, 15]:
    nu_max = 1.0 / m
    rho_c = nu_max / (1 + nu_max)              # = 1/(m+1)
    for rho in [rho_c - 0.05, rho_c + 0.05]:
        z_sim = simulate_flow(rho, nu_max)
        z_star = np.log(rho / ((1 - rho) * nu_max))    # closed form, eq. (7)
        ok = np.sign(z_sim) == np.sign(rho - rho_c)
        print(f"m={m:2d} rho_c={rho_c:.4f} rho={rho:.4f}  simulated z={z_sim:8.3f}  closed-form z*={z_star:8.3f}  "
              f"sign matches predicted side of rho_c: {ok}")
        assert ok
        assert abs(z_sim - z_star) < 1e-3
print("Simulated gradient flow crosses zero exactly where Proposition 1 predicts, for all three m.")

m= 1 rho_c=0.5000 rho=0.4500  simulated z=  -0.201  closed-form z*=  -0.201  sign matches predicted side of rho_c: True


m= 1 rho_c=0.5000 rho=0.5500  simulated z=   0.201  closed-form z*=   0.201  sign matches predicted side of rho_c: True
m= 3 rho_c=0.2500 rho=0.2000  simulated z=  -0.288  closed-form z*=  -0.288  sign matches predicted side of rho_c: True
m= 3 rho_c=0.2500 rho=0.3000  simulated z=   0.251  closed-form z*=   0.251  sign matches predicted side of rho_c: True
m=15 rho_c=0.0625 rho=0.0125  simulated z=  -1.661  closed-form z*=  -1.661  sign matches predicted side of rho_c: True
m=15 rho_c=0.0625 rho=0.1125  simulated z=   0.643  closed-form z*=   0.643  sign matches predicted side of rho_c: True
Simulated gradient flow crosses zero exactly where Proposition 1 predicts, for all three m.


## Part 2 - Full causal Transformer, matched supervision

**This section is an illustrative toy reproduction of the *qualitative phenomenon* in Section 4, not a faithful replication of the paper's exact experiment.** There is no released reference implementation to call into, so everything below is written from scratch and differs from the paper in ways that matter if you want paper-exact numbers, not just the qualitative gap:

- architecture: `nn.TransformerEncoderLayer` with its PyTorch defaults (post-LN, GELU-family internals), not the paper's specific two-block **pre**-LN construction (Section C.1, Fig. 3)
- data: fresh examples resampled every minibatch (effectively infinite data), not a **fixed pool of 100,000 examples** re-used across a reliability sweep (Section B.6) -- so this notebook cannot reproduce the `rho`-sweep of Section 6.5/Fig. 1 as given
- budget: ~1,000 updates at batch 64, vs. **8,000 updates at batch 128**
- tasks: the state-machine uses generic random permutations rather than the paper's derangements; the register task is a toy mod-7 two-register machine, not the paper's exact construction; the Boolean gate set here has 46 synthesized gates (NOT/CNOT/SWAP/Toffoli enumerated programmatically), not the paper's specific 52
- the trace is serialized as states only, with no gate re-emission, unlike Section I.2's `g1,S1,...,gD,SD` format

For each of three deterministic-execution task families, we build four matched continuations that share the same prompt and the same correct answer (eq. 6) and differ only in whether - and how validly - the intermediate computation is supervised:

- **outcome**: prompt -> answer
- **answer_first**: prompt -> answer -> valid trace (trace is causally *after* the answer, so it cannot help predict it)
- **process**: prompt -> valid trace -> answer
- **corrupted**: prompt -> locally-invalid trace -> answer (still ends in the correct answer)

What *does* carry over faithfully: the matched-supervision construction (same prompt/answer across all four conditions, eq. 6), and the causal-masking logic that makes `answer_first`'s trailing trace unusable for the answer itself. The qualitative finding -- process supervision alone teaches execution -- is what Part 1's exact theory predicts, and it is what this section is meant to demonstrate, not a number-for-number match to Table 1.

In [12]:
def register_task(D, N=7, **_):
    K, num_ops = N * N, 5
    x, y = rng.integers(0, N, 2)
    ops, xs, ys = [], [x], [y]
    for _ in range(D):
        op = rng.integers(0, 5)
        if op == 0: x, y = (x + y) % N, y
        elif op == 1: x, y = x, (x + y) % N
        elif op == 2: x, y = y, x
        elif op == 3: x, y = (x + 1) % N, y
        else: x, y = x, (y + 1) % N
        ops.append(op); xs.append(x); ys.append(y)
    states = [xx * N + yy for xx, yy in zip(xs, ys)]
    return [], K, num_ops, states[0], ops, states

def state_machine_task(D, K=8, **_):
    A, B = rng.permutation(K), rng.permutation(K)
    s = int(rng.integers(0, K)); ops, states = [], [s]
    for _ in range(D):
        op = int(rng.integers(0, 2)); s = int((A if op == 0 else B)[s])
        ops.append(op); states.append(s)
    return list(A) + list(B), K, 2, states[0], ops, states

def _build_boolean_gates():
    bit = lambda s, i: (s >> i) & 1
    setb = lambda s, i, v: (s & ~(1 << i)) | (v << i)
    gates = [np.array([s ^ (1 << i) for s in range(16)]) for i in range(4)]
    gates += [np.array([s ^ ((1 << t) if bit(s, c) else 0) for s in range(16)])
              for c in range(4) for t in range(4) if c != t]
    gates += [np.array([setb(setb(s, i, bit(s, j)), j, bit(s, i)) for s in range(16)])
              for i in range(4) for j in range(i + 1, 4)]
    gates += [np.array([s ^ ((1 << t) if bit(s, c1) and bit(s, c2) else 0) for s in range(16)])
              for c1 in range(4) for c2 in range(4) for t in range(4) if len({c1, c2, t}) == 3]
    return gates

BOOL_GATES = _build_boolean_gates()

def boolean_task(D, **_):
    K, num_ops = 16, len(BOOL_GATES)
    s = int(rng.integers(0, K)); ops, states = [], [s]
    for _ in range(D):
        g = int(rng.integers(0, num_ops)); s = int(BOOL_GATES[g][s])
        ops.append(g); states.append(s)
    return [], K, num_ops, states[0], ops, states

In [13]:
def make_sequences(task_fn, D, K):
    table, K, num_ops, s0, ops, states = task_fn(D, K=K)
    COLON, SEP = K + num_ops, K + num_ops + 1
    V = K + num_ops + 2
    prompt = table + [s0] + ops + [SEP]
    P = len(prompt)
    y = states[-1]
    valid_trace = states[1:]
    corrupted_trace = [int(rng.integers(0, K - 1)) for _ in range(D)]
    corrupted_trace = [c if c < states[t + 1] else c + 1 for t, c in enumerate(corrupted_trace)]  # != true successor
    seqs = {
        'outcome':      prompt + [COLON, y],
        'answer_first': prompt + [COLON, y] + valid_trace,
        'process':      prompt + valid_trace + [COLON, y],
        'corrupted':    prompt + corrupted_trace + [COLON, y],
    }
    # the reported "answer" always sits right after COLON; for process/corrupted that's also the last token, but
    # for answer_first it is NOT the last token (the valid trace follows it) -- must track its position explicitly.
    ans_pos = {c: (P + 1 if c in ('outcome', 'answer_first') else len(s) - 1) for c, s in seqs.items()}
    return seqs, ans_pos, P, y, V, K

def batch_of(task_fn, D, K, n, cond):
    rows, P_, V_, K_, A_ = [], None, None, None, None
    for _ in range(n):
        seqs, ans_pos, P, y, V, Kt = make_sequences(task_fn, D, K)
        rows.append(seqs[cond]); P_, V_, K_, A_ = P, V, Kt, ans_pos[cond]
    return torch.tensor(rows, dtype=torch.long), P_, V_, K_, A_

In [14]:
class GPT(nn.Module):
    def __init__(self, V, d=64, H=4, L=2, ff=256, maxlen=64):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(maxlen, d)
        layer = nn.TransformerEncoderLayer(d, H, ff, batch_first=True)
        self.enc, self.head = nn.TransformerEncoder(layer, L), nn.Linear(d, V)
    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.shape[1], device=x.device))
        mask = nn.Transformer.generate_square_subsequent_mask(x.shape[1]).to(x.device)
        return self.head(self.enc(h, mask=mask, is_causal=True))

@torch.no_grad()
def generate(model, prompt, n_new):
    seq = prompt
    for _ in range(n_new):
        nxt = model(seq)[:, -1].argmax(-1, keepdim=True)
        seq = torch.cat([seq, nxt], dim=1)
    return seq

In [15]:
def train_and_eval(task_fn, D, K, cond, steps=1000, batch=64, lr=1e-3, eval_n=256):
    model, opt = None, None
    for step in range(steps):
        batch_x, P, V, _, _ = batch_of(task_fn, D, K, batch, cond)
        batch_x = batch_x.to(device)
        if opt is None:
            model = GPT(V).to(device); opt = torch.optim.AdamW(model.parameters(), lr=lr)
        logits = model(batch_x[:, :-1])
        loss = nn.functional.cross_entropy(logits[:, P - 1:].reshape(-1, V), batch_x[:, P:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    correct, total, Keff = 0, 0, K            # held-out answer accuracy: greedy generation, read off at ans_pos
    for _ in range(eval_n // batch):
        batch_x, P, V, Keff, ans_pos = batch_of(task_fn, D, K, batch, cond)
        y = batch_x[:, ans_pos].to(device)
        out = generate(model, batch_x[:, :P].to(device), batch_x.shape[1] - P)
        correct += (out[:, ans_pos] == y).sum().item(); total += batch
    return 100 * correct / total, 100 / Keff

In [16]:
TASK_CONFIG = {'register': dict(fn=register_task, D=4, K=7),
               'state_machine': dict(fn=state_machine_task, D=4, K=8),
               'boolean': dict(fn=boolean_task, D=8, K=16)}
t0 = time.time()
print(f"{'task':<14}{'condition':<14}{'answer acc %':>13}{'chance %':>10}")
for name, cfg in TASK_CONFIG.items():
    for cond in ['outcome', 'answer_first', 'process', 'corrupted']:
        acc, chance = train_and_eval(cfg['fn'], cfg['D'], cfg['K'], cond)
        print(f"{name:<14}{cond:<14}{acc:13.2f}{chance:10.2f}   ({time.time()-t0:.0f}s)")

task          condition      answer acc %  chance %


register      outcome                8.20      2.04   (11s)
register      answer_first           4.30      2.04   (22s)
register      process               97.27      2.04   (38s)
register      corrupted              3.12      2.04   (50s)
state_machine outcome               15.23     12.50   (63s)
state_machine answer_first          16.02     12.50   (73s)
state_machine process               94.14     12.50   (83s)
state_machine corrupted             12.89     12.50   (94s)
boolean       outcome               19.14      6.25   (105s)
boolean       answer_first          13.28      6.25   (116s)
boolean       process               86.72      6.25   (129s)
boolean       corrupted             10.94      6.25   (140s)


## Scaling up

To approach the paper's exact numbers: increase `K, D` in `TASK_CONFIG`; generate a fixed pool of 100k examples per task (rather than resampling every batch) so a `rho` (trace-reliability) sweep is meaningful; the Transformer here is already the paper's width (128 -> here 64 for speed, bump `d=128`) and depth (2 blocks, 4 heads); train for 8,000 updates at batch 128. Section 6.5's reliability-boundary sweep (`rho_c = 1/(m+1)`, verified analytically in Part 1) can then be reproduced empirically by mixing valid/corrupted traces at controlled ratios inside `make_sequences`.